<font color='darkorange'> Unless otherwise noted, **this notebook will not be reviewed or autograded.**</font> You are welcome to use it for scratchwork, but **only the files listed in the exercises will be checked.**

---

# Exercises

For these exercises, add your functions to the *apputil\.py* file and *app\.py* file as instructed. *These exercises use the same [Titanic dataset](https://www.kaggle.com/competitions/titanic/data) as the lab.*

## Loading the Dataset

When you load your data for this exercise, **make sure to rename columns into lowercase-underscore format.** Write a routine that would rename text like "Age Group" into "age_group". For example, in this data, the column "Pclass" should be renamed to "pclass". *This is a good habit to get into going forward!*


## Exercise 1: Survival Patterns


For this exercise you will analyze survival patterns on the Titanic by looking at passenger class, sex, and age group. Name the function `survival_demographics()` (with no arguments).

1. Create a new column in the Titanic dataset that classifies passengers into age categories (i.e., a pandas `category` series). The categories (case sensitive) should be:
    - `"Child"` (up to 12)
    - `"Teen"` (13–19)
    - `"Adult"` (20–59)
    - `"Senior"` (60+)  
  
	Hint: The `pd.cut()` function might come in handy here.

2. Group the passengers by class, sex, and age group.  

3. For each group, calculate:  
    - The total number of passengers, `n_passengers`
    - The number of survivors, `n_survivors`
    - The survival rate, `survival_rate`

4. Return a DataFrame that includes the results for *all* combinations of class, sex, and age group.  

5. Order the results so they are easy to interpret.  

6. Come up with a clear question that your results table makes you curious about (e.g., “Did women in first class have a higher survival rate than men in other classes?”). Write this question in your `app.py` file above the call to your visualization function, using `st.write("Your Question Here")`.
   
7. Create a Plotly visualization in a function named `visualize_demographic()` that directly addresses your question by returning a Plotly figure (e.g., `fig = px. ...`). You are free to choose the chart type that you think best communicates the findings. Be creative — try different approaches, compare them, and ensure that your chart clearly answers the question you posed.


In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('https://raw.githubusercontent.com/leontoddjohnson/datasets/'
                 'main/data/titanic.csv')
df.columns = df.columns.str.strip().str.replace(r'[ -]+', '_', regex=True).str.lower()

df['age_group'] = pd.cut(df['age'], bins=[-1, 12, 19, 59, 120],
                         labels=['Child', 'Teen', 'Adult', 'Senior'])

demographics = (df.groupby(['pclass', 'sex', 'age_group'], observed=False)
                .agg(n_passengers=('survived', 'size'),
                     n_survivors=('survived', 'sum'))
                .reset_index())
demographics['survival_rate'] = (demographics['n_survivors']
                                 / demographics['n_passengers'])
demographics = demographics.sort_values(['pclass', 'sex', 'age_group'],
                                        ignore_index=True)
display(demographics)

# Did women in first class have a higher survival rate than men in any other class?
fig = px.bar(demographics, x='age_group', y='survival_rate', color='sex',
             barmode='group', facet_col='pclass',
             labels={'age_group': '', 'survival_rate': 'Survival rate', 'sex': ''},
             title='Survival rate by class, sex, and age group')
fig.update_yaxes(tickformat='.0%', range=[0, 1])
fig.update_layout(template='simple_white')
fig.show()

,pclass,sex,age_group,n_passengers,n_survivors,survival_rate
0,1,female,Child,1,0,0.000000
1,1,female,Teen,13,13,1.000000
2,1,female,Adult,68,66,0.970588
3,1,female,Senior,3,3,1.000000
4,1,male,Child,3,3,1.000000
5,1,male,Teen,4,1,0.250000
6,1,male,Adult,80,34,0.425000
7,1,male,Senior,14,2,0.142857
8,2,female,Child,8,8,1.000000
9,2,female,Teen,8,8,1.000000


## Exercise 2: Family Size and Wealth

Using the Titanic dataset, write a function named `family_groups()` to explore the relationship between family size, passenger class, and ticket fare.  

1. Create a new column in the Titanic dataset that represents the total family size for each passenger, `family_size`. Family size is defined as the number of siblings/spouses aboard plus the number of parents/children aboard, plus the passenger themselves.

2. Group the passengers by family size and passenger class. For each group, calculate:  
   - The total number of passengers, `n_passengers`
   - The average ticket fare, `avg_fare`
   - The minimum and maximum ticket fares (to capture variation in wealth), `min_fare` and `max_fare`

3. Return a table with these results, sorted so that the values are clear and easy to interpret (for example, by class and then family size).

4. Write a function called `last_names()` that extracts the last name of each passenger from the `Name` column, and returns the count for each last name (i.e., a pandas series with last name as index, and count as value). Does this result agree with that of the data table above? Share your findings in your app using `st.write`.

5. Just like you did in Exercise 1, come up with a clear question that your results makes you curious about. Write this question in your app.py file above the call to your visualization function. Then, create a Plotly visualization in a function named `visualize_families()` that directly addresses your question. As in Exercise 1 you are free to choose the chart type that you think best communicates the findings.

In [2]:
df['family_size'] = df['sibsp'] + df['parch'] + 1

families = (df.groupby(['pclass', 'family_size'])
            .agg(n_passengers=('fare', 'size'), avg_fare=('fare', 'mean'),
                 min_fare=('fare', 'min'), max_fare=('fare', 'max'))
            .reset_index()
            .sort_values(['pclass', 'family_size'], ignore_index=True))
display(families)

names = df['name'].str.split(',').str[0].str.strip().value_counts()
display(names.head(10))

# The largest shared last names are third-class families, matching the largest
# family_size values above, but they don't agree exactly: unrelated passengers
# share common names, so last-name counts overstate family size.

# Does the average ticket fare increase with family size within each class?
fig = px.line(families, x='family_size', y='avg_fare', color='pclass', markers=True,
              labels={'family_size': 'Family size', 'avg_fare': 'Average fare',
                      'pclass': 'Class'},
              title='Average fare by family size within each class')
fig.update_layout(template='simple_white')
fig.show()

,pclass,family_size,n_passengers,avg_fare,min_fare,max_fare
0,1,1,109,63.672514,0.0000,512.3292
1,1,2,70,91.848039,29.7000,512.3292
2,1,3,24,95.681075,26.2833,211.5000
3,1,4,7,133.521429,120.0000,151.5500
4,1,5,2,262.375000,262.3750,262.3750
5,1,6,4,263.000000,263.0000,263.0000
6,2,1,104,14.066106,0.0000,73.5000
7,2,2,34,24.682962,11.5000,33.0000
8,2,3,31,31.693819,13.0000,73.5000
9,2,4,13,36.575969,11.5000,65.0000


name
Andersson    9
Sage         7
Johnson      6
Panula       6
Goodwin      6
Skoog        6
Carter       6
Rice         5
Palsson      4
Williams     4
Name: count, dtype: int64

## Bonus Question

Add a new column, `older_passenger`, to the Titanic dataset that indicates whether each passenger’s age is above the median age for *their* passenger class. So, suppose row $x$ is in passenger class 2. Then, a value of `True` at row $x$ would indicate that passenger older than 50% of class 2 passengers, and `False` would indicate that they younger.

- You should use pandas functions to accomplish this.
- The new column should contain Boolean values (True if the age is above the median, False if less than or equal to).
- Return the updated table in the function `determine_age_division()`

Once you’ve created this column, consider how this age division relates to your analysis above. Try to visualize this analysis in Plotly using the function name `visualize_age_division()`.

In [3]:
df['older_passenger'] = df['age'] > df.groupby('pclass')['age'].transform('median')
display(df[['pclass', 'age', 'older_passenger']].head())

# Within each class, did passengers older than their class median survive less often?
age_division = (df.groupby(['pclass', 'older_passenger'])['survived']
                .mean()
                .reset_index())
age_division['older_passenger'] = age_division['older_passenger'].map(
    {True: 'Above class median', False: 'At or below'})

fig = px.bar(age_division, x='pclass', y='survived', color='older_passenger',
             barmode='group',
             labels={'pclass': 'Class', 'survived': 'Survival rate',
                     'older_passenger': ''},
             title='Survival rate by age relative to class median age')
fig.update_xaxes(tickvals=[1, 2, 3])
fig.update_yaxes(tickformat='.0%', range=[0, 1])
fig.update_layout(template='simple_white')
fig.show()

,pclass,age,older_passenger
0,3,22.0,False
1,1,38.0,True
2,3,26.0,True
3,1,35.0,False
4,3,35.0,True


## Original Question

**Was survival most predicted by class, sex, or age?**

Each of the three is a categorical predictor of a 0/1 outcome, so they can be
compared on the same scale: the share of the variation in `survived` that each
one explains on its own (eta-squared), and how accurately you could guess a
passenger's fate knowing only that one variable. Rows with a missing `age` are
dropped so all three predictors are scored on the identical 714 passengers.

In [4]:
compare = df.dropna(subset=['age']).copy()
compare['age_group'] = pd.cut(compare['age'], bins=[-1, 12, 19, 59, 120],
                              labels=['Child', 'Teen', 'Adult', 'Senior'])

base = compare['survived'].mean()
ss_total = ((compare['survived'] - base) ** 2).sum()

results = []
for variable in ['pclass', 'sex', 'age_group']:
    groups = compare.groupby(variable, observed=True)['survived'].agg(['size', 'mean'])
    eta_sq = (groups['size'] * (groups['mean'] - base) ** 2).sum() / ss_total
    accuracy = ((groups['size'] * groups['mean'].clip(lower=1 - groups['mean'])).sum()
                / len(compare))
    results.append({'variable': variable,
                    'eta_sq': eta_sq,
                    'accuracy': accuracy,
                    'rate_spread': groups['mean'].max() - groups['mean'].min()})

results = (pd.DataFrame(results)
           .sort_values('eta_sq', ascending=False, ignore_index=True))
display(results)
print(f'baseline accuracy (always guess "died"): {max(base, 1 - base):.3f}')

# Sex explains the most by a wide margin, and it still dominates inside every class:
display(compare.groupby(['pclass', 'sex'], observed=True)['survived']
        .mean().unstack().round(3))

fig = px.bar(results, x='variable', y='eta_sq', color='variable',
             text=results['eta_sq'].map('{:.1%}'.format),
             labels={'variable': '', 'eta_sq': 'Share of survival variation explained'},
             title='Sex predicts Titanic survival far better than class or age')
fig.update_yaxes(tickformat='.0%')
fig.update_layout(template='simple_white', showlegend=False)
fig.show()

,variable,eta_sq,accuracy,rate_spread
0,sex,0.290333,0.780112,0.549491
1,pclass,0.130114,0.675070,0.416477
2,age_group,0.015773,0.609244,0.310479


baseline accuracy (always guess "died"): 0.594


sex,female,male
pclass,,
1,0.965,0.396
2,0.919,0.152
3,0.461,0.150
